In [59]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import pyautogui
import pyperclip
from PyQt5.QtWidgets import *
from PyQt5 import uic
import sys
from datetime import datetime
from datetime import timedelta

# 크롬 드라이버 자동 업데이트
from webdriver_manager.chrome import ChromeDriverManager

# 브라우저 꺼짐 방지
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)

# 불필요한 에러 메시지 없애기
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])

service = Service(executable_path=ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=chrome_options)

# 웹페이지 해당 주소 이동
driver.implicitly_wait(5)
driver.maximize_window()



In [60]:

# 웹 페이지 열기
driver.get("https://admin.blog.naver.com/tmddus341")

time.sleep(1)

# 아이디 입력
id = driver.find_element(By.CSS_SELECTOR, "#id")
id.click()
pyperclip.copy("tmddus341")
pyautogui.hotkey("ctrl", "v")


# 비밀번호 입력
pw = driver.find_element(By.CSS_SELECTOR, "#pw")
pw.click()
pyperclip.copy("thtk1513**")
pyautogui.hotkey("ctrl", "v")


# 로그인 버튼
login_btn = driver.find_element(By.CSS_SELECTOR, "#log\.login")
login_btn.click()




In [61]:
# 서로이웃신청 전체수락 과정
        
driver.find_element(By.CSS_SELECTOR, "#buddylist_config_anchor").click()

print("내가 추가한 이웃 클릭")

# 이웃그룹 선택하기 위해 iframe 접속
driver.switch_to.frame("papermain")
driver.find_element(By.CSS_SELECTOR, "#wrap > ul > li._nclk\(bas_neimgr\.gnei\) > a").click()
print("이웃그룹 클릭")

time.sleep(2)

# 이웃수 리스트
neighbor_num_list = driver.find_elements(By.CSS_SELECTOR, ".num")[ : : 2]

neighbor_index = 0

# 이웃수 500미만 탐색
for neighbor_num in neighbor_num_list:
    neighbor_num = int(neighbor_num.text)

    if neighbor_num < 500:
        break
    else:
        neighbor_index += 1

print(f"이웃 인덱스 : {neighbor_index}")

# iframe 탈출
driver.switch_to.default_content()

driver.find_element(By.CSS_SELECTOR, "#buddyinvite_config_anchor").click()
print("서로이웃 신청 메뉴 클릭")

time.sleep(2)

driver.switch_to.frame("papermain")


내가 추가한 이웃 클릭
이웃그룹 클릭
이웃 인덱스 : 0
서로이웃 신청 메뉴 클릭


In [62]:
# 보낸신청 메뉴 클릭
driver.find_element(By.CSS_SELECTOR, "#inviteMe > ul > li._nclk\(bas_neitadd\.send\) > a").click()

try:
    # 신청일 리스트
    dates = driver.find_elements(By.CSS_SELECTOR, ".date")
except:
    print("보낸 신청이 없습니다.")
    

# date의 text를 담아두는 리스트
date_list = []

for i in range(len(dates)):
    date_list.append(dates[i].text.replace(".", "-")[:-1])

# 신청한 사람 리스트
users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

# 신청취소 버튼 리스트
cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

current_date_time = datetime.now()

# 2주(14일) 전의 날짜 계산
two_weeks_ago = current_date_time - timedelta(days=14)

# 날짜 형식화 (예: YYYY-MM-DD)
two_weeks_ago = two_weeks_ago.strftime("%Y-%m-%d")[2:]

# 6개월 전 날짜 계산
six_months_ago = current_date_time - timedelta(days=180)
six_months_ago = six_months_ago.strftime("%Y-%m-%d")[2:]

a = driver.find_elements(By.CSS_SELECTOR, ".paginate > a")
a_index = 0

########################### 다음 페이지들이 있는 경우 ###########################
if len(a) > 0:

    while 1:

        print("다음 페이지로")

        try:
            # 신청일 리스트
            dates = driver.find_elements(By.CSS_SELECTOR, ".date")
        except:
            print("보낸 신청이 없습니다.")
            break
        

        # date의 text를 담아두는 리스트
        date_list = []

        for index in range(len(dates)):
            date_list.append(dates[index].text.replace(".", "-")[:-1])

        # 신청한 사람 리스트
        users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

        # 신청취소 버튼 리스트
        cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

        i = 0

        while i != len(dates) or i < len(dates):

            print(f"user : {users[i].text}, date : {date_list[i]}, 2주 전 날짜 : {two_weeks_ago}", date_list[i] <= two_weeks_ago)
            
            # 2주 이상 지난 경우 (비교 대상 = <신청일> vs <현재 날짜에서 - 2주전>)
            if date_list[i] <= two_weeks_ago:
                print("2주 이상 지남 -> 삭제 대상")

                t = int(driver.find_element(By.CSS_SELECTOR, ".paginate > strong").text)

                # 신청 취소
                cancels[i].click()
                time.sleep(0.5)

                # 경고창 확인
                result = driver.switch_to.alert
                result.accept()
                result.dismiss()

                # 현재 페이지로 회귀
                if t > 1:
                    print("1페이지가 아님")
                    time.sleep(1.5)
                    a = driver.find_elements(By.CSS_SELECTOR, ".paginate > a")
                    a[a_index-1].click()
                    time.sleep(1.5)


                # 요소들 새로고침
                # 신청일 리스트
                dates = driver.find_elements(By.CSS_SELECTOR, ".date")
                # date의 text를 담아두는 리스트
                date_list = []

                for index in range(len(dates)):
                    date_list.append(dates[index].text.replace(".", "-")[:-1])

                # 신청한 사람 리스트
                users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

                # 신청취소 버튼 리스트
                cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

                continue
                


            # 2주 이상 되지 않은 경우 -> 블로그 들어가서 활동중인 블로그인지 체크
            else:
                # 블로그 6개월 활동 확인
                users[i].click()

                tabs = driver.window_handles
                driver.switch_to.window(tabs[1])
                time.sleep(1)
                url = driver.current_url

                # url 모바일로 변경
                url = url[:8] + "m." + url[8:]

                driver.get(url)
                time.sleep(4)

                try:
                    # 블로그 정렬 바꾸기
                    driver.find_element(By.CSS_SELECTOR, "#postlist_block > div.post_block__Q6T_o > div > div > button:nth-child(3)").send_keys(Keys.ENTER)
                    time.sleep(1)
                
                except:
                    print("비정상 블로그 -> 삭제")
                    driver.close()
                    driver.switch_to.window(tabs[0])

                    driver.switch_to.frame("papermain")

                    t = int(driver.find_element(By.CSS_SELECTOR, ".paginate > strong").text)
                    
                    # 신청 취소
                    cancels[i].send_keys(Keys.ENTER)
                    time.sleep(0.5)

                    # 경고창 확인
                    result = driver.switch_to.alert
                    result.accept()
                    result.dismiss()

                    # 현재 페이지로 회귀
                    if t > 1:
                        print("1페이지가 아님")
                        time.sleep(1.5)
                        a = driver.find_elements(By.CSS_SELECTOR, ".paginate > a")
                        a[a_index-1].click()
                        time.sleep(1.5)

                    # 요소들 새로고침
                    # 신청일 리스트
                    dates = driver.find_elements(By.CSS_SELECTOR, ".date")
                    # date의 text를 담아두는 리스트
                    date_list = []

                    for index in range(len(dates)):
                        date_list.append(dates[index].text.replace(".", "-")[:-1])

                    # 신청한 사람 리스트
                    users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

                    # 신청취소 버튼 리스트
                    cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

                    continue


                # 블로그 최신글 날짜 추출
                blog_time = driver.find_element(By.CSS_SELECTOR, ".time__MHDWV").text
                blog_time = blog_time.replace(". ", "-")[2 : -1]

                print(f"blog_time : {blog_time}, six_moths_age : {six_months_ago}", blog_time < six_months_ago)

                # 6개월전 색출 (비교 대상 = 현재 날짜에서 6개월 전 vs 블로그 최신글 날짜)
                # 먼저 hh시간전, mm분인 블로그 색출
                if "시간" in blog_time or "분" in blog_time:
                    print("hh시간전, mm분인 블로그 -> 유지")
                    driver.close()
                    driver.switch_to.window(tabs[0])
                    time.sleep(2)
                    driver.switch_to.frame("papermain")
                    i += 1
                    continue


                
                # 6개월전 블로그 색출
                else:

                    # 최신글이 6개월 이상 된 경우
                    if blog_time < six_months_ago:
                        print("6개월 이상됨 -> 삭제 대상")
                        driver.close()
                        driver.switch_to.window(tabs[0])
                        time.sleep(2)

                        # 현재 페이지 저장
                        t = int(driver.find_element(By.CSS_SELECTOR, ".paginate > strong").text)

                        # 신청 취소
                        cancels[i].send_keys(Keys.ENTER)
                        time.sleep(0.5)

                        # 경고창 확인
                        result = driver.switch_to.alert
                        result.accept()
                        result.dismiss()

                        driver.switch_to.window(tabs[0])
                        time.sleep(0.5)
                        driver.switch_to.frame("papermain")

                        # 현재 페이지로 회귀
                        if t > 1:
                            print("1페이지가 아님")
                            time.sleep(1.5)
                            a = driver.find_elements(By.CSS_SELECTOR, ".paginate > a")
                            a[a_index-1].click()
                            time.sleep(1.5)

                        # 요소들 새로고침
                        # 신청일 리스트
                        dates = driver.find_elements(By.CSS_SELECTOR, ".date")
                        # date의 text를 담아두는 리스트
                        date_list = []

                        for index in range(len(dates)):
                            date_list.append(dates[index].text.replace(".", "-")[:-1])

                        # 신청한 사람 리스트
                        users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

                        # 신청취소 버튼 리스트
                        cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

                        continue

                    # 최신글이 6개월이 안됨 = 활동중인 블로그 -> 삭제할 필요 없음
                    else:
                        print("활동중인 블로그 -> 유지")
                        driver.close()
                        driver.switch_to.window(tabs[0])
                        time.sleep(2)
                        driver.switch_to.frame("papermain")
                        i += 1
                        continue

        # 다음 페이지들의 리스트
        a = driver.find_elements(By.CSS_SELECTOR, ".paginate > a")

        try:
            a[a_index].click()
            time.sleep(2)
            a_index += 1
            continue

        except:
            print("페이지 끝")
            break


        


########################### 현재 페이지가 끝인 경우 ########################
else:
    print("현재 페이지가 마지막입니다.")

    try:
        # 신청일 리스트
        dates = driver.find_elements(By.CSS_SELECTOR, ".date")
    except:
        print("보낸 신청이 없습니다.")
        
    

    # date의 text를 담아두는 리스트
    date_list = []

    for index in range(len(dates)):
        date_list.append(dates[index].text.replace(".", "-")[:-1])

    # 신청한 사람 리스트
    users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

    # 신청취소 버튼 리스트
    cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

    i = 0

    while i != len(dates) or i < len(dates):

        print(f"user : {users[i].text}, date : {date_list[i]}, 2주 전 날짜 : {two_weeks_ago}", date_list[i] <= two_weeks_ago)
        
        # 2주 이상 지난 경우 (비교 대상 = <신청일> vs <현재 날짜에서 - 2주전>)
        if date_list[i] <= two_weeks_ago:
            print("2주 이상 지남 -> 삭제 대상")

            t = int(driver.find_element(By.CSS_SELECTOR, ".paginate > strong").text)

            time.sleep(2)
            # 신청 취소
            cancels[i].click()
            time.sleep(0.5)

            # 경고창 확인
            result = driver.switch_to.alert
            result.accept()
            result.dismiss()

            # 현재 페이지로 회귀
            if t > 1:
                print("1페이지가 아님")
                time.sleep(1.5)
                a = driver.find_elements(By.CSS_SELECTOR, ".paginate > a")
                a[a_index-1].click()
                time.sleep(1.5)
                

            # 요소들 새로고침
            # 신청일 리스트
            dates = driver.find_elements(By.CSS_SELECTOR, ".date")
            # date의 text를 담아두는 리스트
            date_list = []

            for index in range(len(dates)):
                date_list.append(dates[index].text.replace(".", "-")[:-1])

            # 신청한 사람 리스트
            users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

            # 신청취소 버튼 리스트
            cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

            continue
            


        # 2주 이상 되지 않은 경우 -> 블로그 들어가서 활동중인 블로그인지 체크
        else:
            # 블로그 6개월 활동 확인
            users[i].click()

            tabs = driver.window_handles
            driver.switch_to.window(tabs[1])
            time.sleep(1)
            url = driver.current_url

            # url 모바일로 변경
            url = url[:8] + "m." + url[8:]

            driver.get(url)
            time.sleep(4)

            try:
                # 블로그 정렬 바꾸기
                driver.find_element(By.CSS_SELECTOR, "#postlist_block > div.post_block__Q6T_o > div > div > button:nth-child(3)").send_keys(Keys.ENTER)
                time.sleep(1)
            
            except:
                print("비정상 블로그 -> 삭제")
                driver.close()
                driver.switch_to.window(tabs[0])

                driver.switch_to.frame("papermain")

                t = int(driver.find_element(By.CSS_SELECTOR, ".paginate > strong").text)
                
                # 신청 취소
                cancels[i].send_keys(Keys.ENTER)
                time.sleep(0.5)

                # 경고창 확인
                result = driver.switch_to.alert
                result.accept()
                result.dismiss()

                # 현재 페이지로 회귀
                if t > 1:
                    time.sleep(1.5)
                    a = driver.find_elements(By.CSS_SELECTOR, ".paginate > a")
                    print("1페이지가 아님")
                    a[a_index-1].click()
                    time.sleep(1.5)

                # 요소들 새로고침
                # 신청일 리스트
                dates = driver.find_elements(By.CSS_SELECTOR, ".date")
                # date의 text를 담아두는 리스트
                date_list = []

                for index in range(len(dates)):
                    date_list.append(dates[index].text.replace(".", "-")[:-1])

                # 신청한 사람 리스트
                users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

                # 신청취소 버튼 리스트
                cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

                continue


            # 블로그 최신글 날짜 추출
            blog_time = driver.find_element(By.CSS_SELECTOR, ".time__MHDWV").text
            blog_time = blog_time.replace(". ", "-")[2 : -1]

            print(f"blog_time : {blog_time}, six_moths_age : {six_months_ago}", blog_time < six_months_ago)

            # 6개월전 색출 (비교 대상 = 현재 날짜에서 6개월 전 vs 블로그 최신글 날짜)
            # 먼저 hh시간전, mm분인 블로그 색출
            if "시간" in blog_time or "분" in blog_time:
                print("hh시간전, mm분인 블로그 -> 유지")
                driver.close()
                driver.switch_to.window(tabs[0])
                time.sleep(2)
                driver.switch_to.frame("papermain")
                i += 1
                continue


            
            # 6개월전 블로그 색출
            else:

                # 최신글이 6개월 이상 된 경우
                if blog_time < six_months_ago:
                    print("6개월 이상됨 -> 삭제 대상")
                    driver.close()
                    driver.switch_to.window(tabs[0])
                    time.sleep(2)

                    t = int(driver.find_element(By.CSS_SELECTOR, ".paginate > strong").text)

                    # 신청 취소
                    cancels[i].click()
                    time.sleep(0.5)

                    # 경고창 확인
                    result = driver.switch_to.alert
                    result.accept()
                    result.dismiss()

                    driver.switch_to.window(tabs[0])
                    time.sleep(0.5)
                    driver.switch_to.frame("papermain")

                    # 현재 페이지로 회귀
                    if t > 1:
                        print("1페이지가 아님")
                        time.sleep(1.5)
                        a = driver.find_elements(By.CSS_SELECTOR, ".paginate > a")
                        a[a_index-1].click()
                        time.sleep(1.5)

                    # 요소들 새로고침
                    # 신청일 리스트
                    dates = driver.find_elements(By.CSS_SELECTOR, ".date")
                    # date의 text를 담아두는 리스트
                    date_list = []

                    for index in range(len(dates)):
                        date_list.append(dates[index].text.replace(".", "-")[:-1])

                    # 신청한 사람 리스트
                    users = driver.find_elements(By.CSS_SELECTOR, ".nickname")

                    # 신청취소 버튼 리스트
                    cancels = driver.find_elements(By.CSS_SELECTOR, ".btn.btn5")

                    continue

                # 최신글이 6개월이 안됨 = 활동중인 블로그 -> 삭제할 필요 없음
                else:
                    print("활동중인 블로그 -> 유지")
                    driver.close()
                    driver.switch_to.window(tabs[0])
                    time.sleep(2)
                    driver.switch_to.frame("papermain")
                    i += 1
                    continue





            











현재 페이지가 마지막입니다.
user : 차디러루너, date : 23-09-24, 2주 전 날짜 : 23-10-16 True
2주 이상 지남 -> 삭제 대상
user : Erin쌤, date : 23-09-24, 2주 전 날짜 : 23-10-16 True
2주 이상 지남 -> 삭제 대상
user : 마시치요, date : 23-09-24, 2주 전 날짜 : 23-10-16 True
2주 이상 지남 -> 삭제 대상
user : 카보틴, date : 17-08-25, 2주 전 날짜 : 23-10-16 True
2주 이상 지남 -> 삭제 대상


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=118.0.5993.118)
Stacktrace:
	GetHandleVerifier [0x00504DE3+43907]
	(No symbol) [0x00490741]
	(No symbol) [0x003833ED]
	(No symbol) [0x0036A9F2]
	(No symbol) [0x003D68CB]
	(No symbol) [0x003E5103]
	(No symbol) [0x003D2956]
	(No symbol) [0x003AE17E]
	(No symbol) [0x003AF32D]
	GetHandleVerifier [0x007B5AF9+2865305]
	GetHandleVerifier [0x007FE78B+3163435]
	GetHandleVerifier [0x007F8441+3138017]
	GetHandleVerifier [0x0058E0F0+605840]
	(No symbol) [0x0049A64C]
	(No symbol) [0x00496638]
	(No symbol) [0x0049675F]
	(No symbol) [0x00488DB7]
	BaseThreadInitThunk [0x7756FCC9+25]
	RtlGetAppContainerNamedObjectPath [0x77C97C6E+286]
	RtlGetAppContainerNamedObjectPath [0x77C97C3E+238]
